# Linux system performance analysis: /proc, cgroups, and systemd-cgtop

> L3 concept exercise — This notebook combines three Linux performance data sources into one mental model: the kernel's `/proc` pseudo-filesystem for raw counters, the cgroup hierarchy for resource limits and current usage, and `systemd-cgtop` for a live aggregated view. The goal is to show how each layer contributes a different piece of the performance puzzle and how they relate to one another.


## Purpose

A production Linux host exposes performance data at three distinct layers. `/proc` gives immediate access to kernel counters (CPU jiffies, memory free vs. used, uptime) without requiring any extra package. cgroups provide the resource-control layer: each slice or scope has a maximum and a current usage that can be read directly from `cpu.max`, `memory.max`, and `io.stat`. `systemd-cgtop` sits above both, aggregating those cgroup metrics into a live, human-readable dashboard. This notebook walks through each layer, reads the data with small Python helpers, and shows how to correlate them.


In [ ]:
# last_verified: 2026-08-11 · linux system administration n/a
import subprocess
from pathlib import Path
import time

CGROUP_ROOT = Path("/sys/fs/cgroup")
PROC_ROOT = Path("/proc")


## /proc — raw kernel counters

The `/proc` filesystem is not a real disk filesystem; it is a kernel interface that exposes process and system information as plain-text files. The most useful files for performance analysis are:

- **`/proc/stat`** — aggregate CPU jiffies since boot (user, system, idle, iowait, steal).
- **`/proc/meminfo`** — memory totals (MemTotal, MemFree, MemAvailable, Buffers, Cached, SwapTotal, SwapFree).
- **`/proc/uptime`** — total uptime in seconds and cumulative idle time.

Reading these files directly is fast and does not require root privileges for the summary lines shown below.


In [ ]:
def read_proc_cpu():
    """Parse /proc/stat and return the first cpu aggregate line as a dict."""
    with open(PROC_ROOT / "stat") as f:
        for line in f:
            if line.startswith("cpu "):
                parts = line.split()
                keys = ["user", "nice", "system", "idle", "iowait", "irq", "softirq", "steal", "guest", "guest_nice"]
                return dict(zip(keys, map(int, parts[1:])))
    return {}


def read_proc_meminfo():
    """Parse /proc/meminfo and return the fields we care about as a dict."""
    mem = {}
    with open(PROC_ROOT / "meminfo") as f:
        for line in f:
            key, rest = line.split(":", 1)
            if key in {"MemTotal", "MemFree", "MemAvailable", "Buffers", "Cached", "SwapTotal", "SwapFree", "Active", "Inactive"}:
                mem[key] = int(rest.strip().split()[0])  # kB
    return mem


def read_proc_uptime():
    """Return (uptime_seconds, idle_seconds) from /proc/uptime."""
    with open(PROC_ROOT / "uptime") as f:
        up, idle = f.read().split()
        return float(up), float(idle)


cpu_before = read_proc_cpu()
mem = read_proc_meminfo()
uptime, idle = read_proc_uptime()

print(f"Uptime: {uptime/3600:.1f} h")
total_mem = mem.get('MemTotal', 0)
avail_mem = mem.get('MemAvailable', 0)
print(f"Memory used: {total_mem - avail_mem} / {total_mem} kB")
total_swap = mem.get('SwapTotal', 0)
free_swap = mem.get('SwapFree', 0)
print(f"Swap used: {total_swap - free_swap} / {total_swap} kB")
print(f"CPU sample (jiffies): user={cpu_before.get('user', 0)} system={cpu_before.get('system', 0)} idle={cpu_before.get('idle', 0)} iowait={cpu_before.get('iowait', 0)}")

## cgroups — resource limits and current usage

cgroups (control groups) are a kernel feature that limits, accounts for, and isolates resource usage of process groups. On systemd hosts, every service and slice maps to a cgroup under `/sys/fs/cgroup`. The key files for performance analysis are:

- **`cpu.max`** — two numbers: quota (microseconds allowed per period) and period (microseconds). If the first number is `max`, the group is unthrottled.
- **`memory.max`** — hard memory limit in bytes, or `max` if unset.
- **`cpu.stat`** — contains `usage_usec` (total CPU time consumed) and `user_usec` / `system_usec`.
- **`io.stat`** — per-device I/O bytes and CPU time.

The cell below walks the cgroup tree, finds the system.slice and the current shell's cgroup, and prints the relevant limits and current usage.


In [ ]:
def read_cgroup_file(path: Path) -> str:
    try:
        return path.read_text().strip()
    except FileNotFoundError:
        return "<not found>"
    except PermissionError:
        return "<permission denied>"


def parse_cpu_max(text: str) -> dict:
    """Parse cpu.max: returns {quota, period, throttling_pct}."""
    parts = text.split()
    if len(parts) < 2:
        return {"quota": None, "period": None, "throttling_pct": None}
    quota_str, period_str = parts[0], parts[1]
    if quota_str == "max":
        return {"quota": None, "period": int(period_str), "throttling_pct": 0.0}
    quota, period = int(quota_str), int(period_str)
    return {"quota": quota, "period": period, "throttling_pct": (quota / period) * 100}


def parse_memory_max(text: str) -> dict:
    if text == "max" or not text:
        return {"limit_bytes": None}
    return {"limit_bytes": int(text)}


# Inspect the current shell cgroup and the root system.slice
my_cgroup = Path("/proc/self/cgroup")
print("--- /proc/self/cgroup ---")
print(read_cgroup_file(my_cgroup))

system_slice = CGROUP_ROOT / "system.slice"
if system_slice.exists():
    cpu_max = read_cgroup_file(system_slice / "cpu.max")
    mem_max = read_cgroup_file(system_slice / "memory.max")
    cpu_stat = read_cgroup_file(system_slice / "cpu.stat")
    print("\n--- system.slice limits ---")
    print(f"cpu.max: {cpu_max}")
    print(f"memory.max: {mem_max}")
    print(f"cpu.stat: {cpu_stat}")

# Also read the current process cgroup path
proc_cgroup = read_cgroup_file(my_cgroup)
if proc_cgroup and proc_cgroup != "<not found>":
    cgroup_path = proc_cgroup.split(":")[-1].strip()
    cgroup_dir = CGROUP_ROOT / cgroup_path
    if cgroup_dir.exists():
        cpu_max = read_cgroup_file(cgroup_dir / "cpu.max")
        mem_max = read_cgroup_file(cgroup_dir / "memory.max")
        print(f"\n--- current process cgroup ({cgroup_path}) ---")
        print(f"cpu.max: {cpu_max}")
        print(f"memory.max: {mem_max}")

## systemd-cgtop — live aggregated view

`systemd-cgtop` reads the same cgroup data but presents it as a live, ranked table grouped by slice, scope, or service. It is useful for spotting which unit is consuming CPU, memory, or I/O without manually traversing `/sys/fs/cgroup`.

Running it non-interactively requires `--batch` and a limited iteration count so the cell does not hang.


In [ ]:
def run_cgtop(iterations: int = 1, delay: float = 0.5) -> str:
    """Run systemd-cgtop in batch mode and return stdout."""
    try:
        result = subprocess.run(
            ["systemd-cgtop", "--batch", "--iterations", str(iterations), "--delay", str(delay)],
            capture_output=True,
            text=True,
            timeout=10,
        )
        return result.stdout.strip()
    except FileNotFoundError:
        return "systemd-cgtop is not installed on this host"
    except subprocess.TimeoutExpired:
        return "systemd-cgtop timed out"


cgtop_output = run_cgtop(iterations=1, delay=0.5)
print(cgtop_output)

## Combine the three views

A typical performance investigation starts with `/proc` to confirm the symptom (e.g., high iowait or low available memory), then drills into cgroups to see whether a specific unit is hitting its limit, and finally uses `systemd-cgtop` to compare units side by side. The cell below computes a quick CPU usage delta from `/proc/stat` and pairs it with the current cgroup CPU quota so the relationship between raw jiffies and throttling is visible.


In [ ]:
def cpu_usage_delta(before: dict, after: dict) -> float:
    """Return CPU usage percentage between two /proc/stat samples."""
    def total(d):
        return sum(d.get(k, 0) for k in d)
    idle = after.get("idle", 0) - before.get("idle", 0)
    iowait = after.get("iowait", 0) - before.get("iowait", 0)
    total_all = total(after) - total(before)
    if total_all == 0:
        return 0.0
    return ((total_all - idle - iowait) / total_all) * 100


# Sample twice with a short sleep to compute usage
print("Sampling CPU for 1 second...")
cpu_before = read_proc_cpu()
time.sleep(1)
cpu_after = read_proc_cpu()
usage_pct = cpu_usage_delta(cpu_before, cpu_after)
print(f"CPU usage over the last second: {usage_pct:.1f}%")

# Pair with cgroup CPU quota if available
my_cgroup_path = None
proc_cgroup = read_cgroup_file(Path("/proc/self/cgroup"))
if proc_cgroup and proc_cgroup != "<not found>":
    my_cgroup_path = proc_cgroup.split(":")[-1].strip()
    cgroup_dir = CGROUP_ROOT / my_cgroup_path
    if cgroup_dir.exists():
        cpu_max = read_cgroup_file(cgroup_dir / "cpu.max")
        parsed = parse_cpu_max(cpu_max)
        if parsed["quota"]:
            q = parsed['quota']
            p = parsed['period']
            t = parsed['throttling_pct']
            print(f"Cgroup CPU quota: {q} / {p} µs ({t:.1f}% of one core)")
        else:
            print("Cgroup CPU quota: unthrottled (max)")

## Verify

After running all cells, confirm that:

1. `/proc/stat` and `/proc/meminfo` returned numeric values without errors.
2. The cgroup cell located both `system.slice` and the current process's cgroup, and printed non-`<not found>` values for `cpu.max` and `memory.max`.
3. `systemd-cgtop` returned a table (or the expected fallback message if the binary is absent).
4. The CPU usage delta cell reported a non-zero percentage.

If any cell raised `PermissionError`, re-run it with elevated privileges or skip the cgroup section — the `/proc` section works for unprivileged users.
